# All-in-one: LightOnOCR + context extraction + Evaluation


**Step 1 — Dual extraction** (same PDFs, OCR once per paper):
- **`heuristic`**: caption + mentions from regex/heuristics on OCR text (no Ollama). GLiNER2 only fills datasets/metrics in the JSON (not evaluated here).
- **`heuristic_ollama`**: same caption heuristics; **Ollama filters mentions** (one run per model in `OLLAMA_MODELS`).

**Step 2 — Mentions evaluation** vs `ground_truth_context.json` (BERTScore best-match). Caption is not scored here — both modes use the same heuristic caption extraction, so it cannot differentiate `heuristic` vs `heuristic_ollama`.

Run from `table_extraction/`. Requires Ollama, LightOnOCR (`transformers`), GLiNER2, `bert-score`.

**Outputs:**
- `table_context_lightonocr_gliner.json` → predictions for **`heuristic`**
- `table_context_lightonocr_gliner_ollama_<model>.json` → predictions for **`heuristic_ollama`**
- `eval_context/context_eval.csv` → macro BERTScore on **mentions** (`mode` = `heuristic` | `heuristic_ollama`)

**Troubleshooting LightOnOCR import:** use the **same Jupyter kernel** as `extract_table_context_lightonocr_gliner.ipynb`. If import fails here only, run `import transformers; print(transformers.__version__, transformers.__file__)` in both notebooks — versions must match (`>=4.57.6` for `LightOnOcrForConditionalGeneration`). Restart kernel after `pip install -U 'transformers>=4.57.6'`.


In [ ]:
# If needed on a clean server, uncomment these installs:
# !pip install -q torch "transformers>=4.57.6" pypdfium2 pillow beautifulsoup4 "gliner2>=1.2.5" bert-score pylatexenc

import json
import os
import re
import tempfile
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import pypdfium2 as pdfium
import pandas as pd
import torch
from PIL import Image
from bs4 import BeautifulSoup
import requests

print("Imports OK")

In [ ]:
# --- Config ---
# Small manual benchmark (caption + mentions + datasets + metrics):
# PDF_DIR = Path("pdfs_test")
# GROUND_TRUTH_PATH = PDF_DIR / "ground_truth_context.json"

# Large PwC-backed benchmark (datasets + metrics; arxiv id as paper key):
PDF_DIR = Path("../data/pdf_files_3")
GROUND_TRUTH_PATH = PDF_DIR / "ground_truth_context.json"

GLINER_OUTPUT_PATH = PDF_DIR / "table_context_lightonocr_gliner.json"
OCR_CACHE_DIR = PDF_DIR / "ocr_cache"
EVAL_OUTPUT_DIR = PDF_DIR / "eval_context"

GLINER_RESULT_JSON_DESCRIPTION = (
    "Per-table context (caption + mentions) with datasets & metrics via GLiNER2 "
    "(same multi-pass extraction as table_*_benchmark_gliner notebooks)."
)
FORCE_OCR = False

# Labels in context_eval.csv (caption/mentions evaluation only):
#   heuristic         — regex/heuristics on OCR text (no Ollama)
#   heuristic_ollama  — same + Ollama mention filter
EVAL_MODE_HEURISTIC = "heuristic"
EVAL_MODE_HEURISTIC_OLLAMA = "heuristic_ollama"

OLLAMA_MODELS = ["qwen3:1.7b", "llama3:8b"]
OLLAMA_URL = "http://localhost:11434/api/generate"
OLLAMA_TIMEOUT = 120
USE_OLLAMA_FOR_CONTEXT = True


def ollama_model_slug(model: str) -> str:
    return model.replace(":", "_").replace(".", "_").replace("/", "_")


def ollama_output_path(model: str) -> Path:
    return PDF_DIR / f"table_context_lightonocr_gliner_ollama_{ollama_model_slug(model)}.json"


def ollama_result_description(model: str) -> str:
    return (
        "Per-table context (caption + mentions) with datasets & metrics via GLiNER2 "
        f"and Ollama mention refinement ({model}, LightOnOCR)."
    )


OCR_MODEL_ID = "lightonai/LightOnOCR-2-1B"
OCR_MAX_NEW_TOKENS = 8192
OCR_TARGET_LONGEST = 1540

GLINER2_MODEL_ID = "fastino/gliner2-base-v1"
GLINER2_MIN_SCORE = 0.65
GLINER2_MAX_CHARS = 3000

ADJACENT_PARA_WINDOW = 2
MIN_MENTION_CHARS = 40

BERTSCORE_LANG = "en"
BERTSCORE_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MENTION_SEP = " [SEP] "

PDF_DIR.mkdir(parents=True, exist_ok=True)
OCR_CACHE_DIR.mkdir(parents=True, exist_ok=True)
EVAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PDF_FILES = sorted(PDF_DIR.glob("*.pdf"))
assert PDF_FILES, f"No PDFs in {PDF_DIR}"

print(f"PDFs: {len(PDF_FILES)} | Ollama models: {OLLAMA_MODELS} | BERTScore device: {BERTSCORE_DEVICE}")


In [ ]:
# Load LightOnOCR (same import as extract_table_context_lightonocr_gliner.ipynb)
import transformers

print(f"transformers {transformers.__version__} @ {transformers.__file__}")

try:
    from transformers import LightOnOcrForConditionalGeneration, LightOnOcrProcessor
except ImportError as exc:
    raise ImportError(
        f"LightOnOCR classes not found in transformers {transformers.__version__}. "
        "Upgrade with: pip install -U 'transformers>=4.57.6' then restart the kernel. "
        "If extract_table_context_lightonocr_gliner.ipynb works, compare "
        "transformers.__file__ in both notebooks — you may be on different kernels."
    ) from exc

if torch.cuda.is_available():
    ocr_device = "cuda"
    ocr_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
elif torch.backends.mps.is_available():
    ocr_device = "mps"
    ocr_dtype = torch.float16
else:
    ocr_device = "cpu"
    ocr_dtype = torch.float32

print(f"OCR device: {ocr_device} | dtype: {ocr_dtype}")

ocr_processor = LightOnOcrProcessor.from_pretrained(OCR_MODEL_ID)
ocr_model = LightOnOcrForConditionalGeneration.from_pretrained(
    OCR_MODEL_ID,
    torch_dtype=ocr_dtype,
    attn_implementation="eager",
).to(ocr_device)

print("LightOnOCR model loaded")

In [ ]:
# Disable PyTorch JIT/TensorExpr fusion (DeBERTa-v3 backbone used by GLiNER2 can trigger
# an nvrtc JIT path that fails on some CUDA builds). Eager kernels are correct + tiny slowdown.
os.environ.setdefault("PYTORCH_JIT", "0")
os.environ.setdefault("PYTORCH_NVFUSER_DISABLE", "1")
for _name, _args in [
    ("_jit_set_profiling_executor", (False,)),
    ("_jit_set_profiling_mode", (False,)),
    ("_jit_override_can_fuse_on_gpu", (False,)),
    ("_jit_override_can_fuse_on_cpu", (False,)),
    ("_jit_set_texpr_fuser_enabled", (False,)),
    ("_jit_set_nvfuser_enabled", (False,)),
]:
    _fn = getattr(torch._C, _name, None)
    if _fn is not None:
        try:
            _fn(*_args)
        except Exception:
            pass

# Load GLiNER2 (same entity extractor used by the metric/dataset notebooks)
from gliner2 import GLiNER2

GLINER2_LABEL_DESCRIPTIONS = {
    "model": (
        "Name of a model, method, or algorithm "
        "(e.g. TransE, ComplEx, RotatE)."
    ),
    "dataset": (
        "Name of a benchmark dataset or knowledge-graph corpus "
        "(e.g. WN18, FB15k, YAGO)."
    ),
    "metric": (
        "Name of an evaluation metric used for reporting performance "
        "(e.g. MRR, Hits@10, F1)."
    ),
}
GLINER2_LABELS = list(GLINER2_LABEL_DESCRIPTIONS.keys())

gliner2_map_location = "cuda" if torch.cuda.is_available() else "cpu"
gliner2_model = GLiNER2.from_pretrained(GLINER2_MODEL_ID, map_location=gliner2_map_location)
print(f"GLiNER2 model loaded: {GLINER2_MODEL_ID} on {gliner2_map_location}")


In [ ]:
# ---- OCR helpers ----------------------------------------------------------
def render_pdf_page(pdf_doc, page_idx: int, target_longest: int = OCR_TARGET_LONGEST) -> Image.Image:
    page = pdf_doc[page_idx]
    img = page.render(scale=200 / 72).to_pil()
    w, h = img.size
    longest = max(w, h)
    if longest > target_longest:
        ratio = target_longest / longest
        img = img.resize((int(w * ratio), int(h * ratio)), Image.LANCZOS)
    return img.convert("RGB") if img.mode != "RGB" else img


def ocr_page(img: Image.Image, max_new_tokens: int = OCR_MAX_NEW_TOKENS) -> str:
    tmp = tempfile.NamedTemporaryFile(suffix=".png", delete=False)
    img.save(tmp, format="PNG")
    tmp.close()
    try:
        conv = [{"role": "user", "content": [{"type": "image", "url": tmp.name}]}]
        inputs = ocr_processor.apply_chat_template(
            conv, add_generation_prompt=True, tokenize=True,
            return_dict=True, return_tensors="pt",
        )
        inputs = {
            k: v.to(device=ocr_device, dtype=ocr_dtype) if v.is_floating_point() else v.to(ocr_device)
            for k, v in inputs.items()
        }
        with torch.no_grad():
            out = ocr_model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
        gen = out[0, inputs["input_ids"].shape[1]:]
        return ocr_processor.decode(gen, skip_special_tokens=True)
    finally:
        os.unlink(tmp.name)


def _pdf_looks_readable(pdf_path: Path) -> tuple[bool, str]:
    """Quick check before Pdfium (corrupt downloads often fail with Data format error)."""
    try:
        size = pdf_path.stat().st_size
    except OSError as e:
        return False, str(e)
    if size < 128:
        return False, f"file too small ({size} bytes)"
    with pdf_path.open("rb") as f:
        head = f.read(8)
    if not head.startswith(b"%PDF-"):
        return False, f"missing %PDF- header (got {head[:40]!r})"
    return True, ""


def ocr_pdf_pages(pdf_path: Path, *, force_ocr: bool = False) -> List[str]:
    """OCR every page of a PDF to text. Cached per PDF as <stem>_pages.json."""
    OCR_CACHE_DIR.mkdir(parents=True, exist_ok=True)
    cache_file = OCR_CACHE_DIR / f"{pdf_path.stem}_pages.json"
    if cache_file.exists() and not force_ocr:
        with cache_file.open(encoding="utf-8") as f:
            return json.load(f)["pages"]

    ok, why = _pdf_looks_readable(pdf_path)
    if not ok:
        raise ValueError(f"unreadable PDF ({why}): {pdf_path.name}")

    try:
        pdf_doc = pdfium.PdfDocument(str(pdf_path))
    except Exception as e:
        raise ValueError(f"Pdfium cannot open {pdf_path.name}: {e}") from e

    pages = []
    try:
        for page_idx in range(len(pdf_doc)):
            print(f"  OCR page {page_idx + 1}/{len(pdf_doc)}", end="\r", flush=True)
            pages.append(ocr_page(render_pdf_page(pdf_doc, page_idx)))
    finally:
        pdf_doc.close()
    print()

    with cache_file.open("w", encoding="utf-8") as f:
        json.dump({"file_name": str(pdf_path), "pages": pages}, f, ensure_ascii=False)
    return pages


# ---- Context patterns -----------------------------------------------------
# A table number: arabic (1, 2.1), roman (IV) or single-letter-prefixed (A1).
_NUM = r"[A-Za-z]?\d+(?:\.\d+)?|[IVXLC]+"

TABLE_BLOCK_RE = re.compile(r"<table\b[^>]*>.*?</table>", re.DOTALL | re.IGNORECASE)
# Caption: "Table N:" / "Table N." / "**Table N**:" at the start of a line/segment.
CAPTION_RE = re.compile(rf"(?:\*\*)?(?:Table|Tab\.?|TABLE)\s+({_NUM})(?:\*\*)?\s*[:.\u2014-]\s+", re.IGNORECASE)
# In-text reference (single or grouped: "Tables 2 and 3").
TABLE_REF_RE = re.compile(rf"\b(?:Table|Tab\.?|TABLE|Tables|TABLES)\s+({_NUM})(?:\s*(?:,|and|&)\s*({_NUM}))*", re.IGNORECASE)
# Strip the leading 'Table(s)' keyword before reading the referenced numbers.
_REF_KEYWORD_RE = re.compile(r"^(?:Tables?|Tab\.?|TABLES?)\s*", re.IGNORECASE)
# Number extractor (no IGNORECASE: avoids matching the lowercase 'l' in 'Table' as roman L).
_REF_NUM_RE = re.compile(r"[A-Za-z]?\d+(?:\.\d+)?|[IVXLC]+")


def normalize_table_number(raw: str) -> str:
    return raw.strip().upper().replace(" ", "")


def _collapse_ws(text: str) -> str:
    return re.sub(r"[ \t]+", " ", text).strip()


def _repair_hyphen_breaks(text: str) -> str:
    """Rejoin hyphenated words split across line breaks (lowercase continuation only)."""
    return re.sub(r"(\w)-\s*\n\s*([a-z][\w'-]*)", r"\1\2", text)


def _split_paragraphs(text: str) -> List[str]:
    """Split page text into paragraphs; fall back to line/sentence chunks when OCR has no blank lines."""
    text = _repair_hyphen_breaks(text)
    paras = [p.strip() for p in re.split(r"\n\s*\n+", text) if p.strip()]
    if len(paras) > 1:
        return [_collapse_ws(p) for p in paras]

    lines = [ln.strip() for ln in text.split("\n") if ln.strip()]
    if len(lines) <= 1:
        return [_collapse_ws(text)] if text.strip() else []

    chunks: List[str] = []
    buf: List[str] = []
    for line in lines:
        buf.append(line)
        if line.endswith((".", "!", "?", '."', '."', '."')):
            chunks.append(" ".join(buf))
            buf = []
    if buf:
        chunks.append(" ".join(buf))
    if len(chunks) > 1:
        return [_collapse_ws(p) for p in chunks if p.strip()]
    return [_collapse_ws(text)] if text.strip() else []


def _table_nums_in_para(para: str) -> set:
    nums: set = set()
    for m in TABLE_REF_RE.finditer(para):
        body = _REF_KEYWORD_RE.sub("", m.group(0))
        for g in _REF_NUM_RE.findall(body):
            nums.add(normalize_table_number(g))
    return nums


def _mention_key(page: int, text: str) -> Tuple[int, str]:
    return page, text


def find_page_captions(page_text: str) -> List[dict]:
    """All caption segments on a page, sorted by start position."""
    captions = []
    for m in CAPTION_RE.finditer(page_text):
        start = m.start()
        rest = page_text[start:]
        para = re.split(r"\n\s*\n", rest, maxsplit=1)[0]
        caption = _collapse_ws(para)
        if not caption:
            continue
        captions.append({
            "table_num": normalize_table_number(m.group(1)),
            "caption": caption,
            "start": start,
            "end": start + len(para),
        })
    return captions


def pair_captions_to_tables(page_text: str) -> List[dict]:
    """Assign one caption to each <table> block on the page.

    When caption and table counts match, pair by document order (fixes multi-table pages
    where table N's caption sits before the block but table N+1's caption sits after it).
    Otherwise fall back to the nearest unused caption.
    """
    blocks = list(TABLE_BLOCK_RE.finditer(page_text))
    captions = find_page_captions(page_text)
    if not blocks:
        return []

    results: List[dict] = []
    if len(captions) == len(blocks):
        caps_sorted = sorted(captions, key=lambda c: c["start"])
        for bm, cap in zip(blocks, caps_sorted):
            results.append({
                "match": bm,
                "caption": cap["caption"],
                "table_num": cap["table_num"],
            })
        return results

    used: set = set()
    for bm in blocks:
        t_start, t_end = bm.start(), bm.end()
        best_idx = None
        best_dist = float("inf")
        for ci, cap in enumerate(captions):
            if ci in used:
                continue
            if cap["end"] <= t_start:
                dist = t_start - cap["end"]
            elif cap["start"] >= t_end:
                dist = cap["start"] - t_end
            else:
                dist = 0
            if dist < best_dist:
                best_dist = dist
                best_idx = ci
        if best_idx is not None:
            used.add(best_idx)
            cap = captions[best_idx]
            results.append({
                "match": bm,
                "caption": cap["caption"],
                "table_num": cap["table_num"],
            })
        else:
            results.append({"match": bm, "caption": "", "table_num": None})
    return results


def page_paragraphs_without_tables(page_text: str) -> List[str]:
    text = TABLE_BLOCK_RE.sub(" ", page_text)
    return _split_paragraphs(text)


_HEADING_ONLY_RE = re.compile(r"^#{1,6}\s+\S")
_FOOTNOTE_URL_RE = re.compile(r"https?://|github\.com|\$\^\{\d+\}")


def _is_safe_continuation(para: str) -> bool:
    t = para.strip()
    if not t:
        return False
    if CAPTION_RE.match(t):
        return False
    if re.match(r"^Table\b", t, re.IGNORECASE):
        return False
    return True


_HYPHEN_CONTINUATION_PREFIX_RE = re.compile(r"^([a-z][a-z'-]{0,30})([.,;:!?])?")
_SENTENCE_END_RE = re.compile(r"""[.!?]['")\]]*\s*$""")
MAX_MENTION_EXTENSIONS = 2


def _ends_complete_sentence(text: str) -> bool:
    return bool(_SENTENCE_END_RE.search(text.rstrip()))


def _continuation_paragraphs(
    page_idx: int,
    para_idx: int,
    paras: List[str],
    pages: List[str],
):
    for k in range(para_idx + 1, len(paras)):
        yield paras[k]
    if page_idx < len(pages):
        for p in page_paragraphs_without_tables(pages[page_idx]):
            yield p


def _extract_hyphen_continuation_prefix(para: str) -> Optional[str]:
    """Leading lowercase word fragment after a hyphenated line break."""
    t = para.lstrip()
    if not t or not t[0].islower():
        return None
    if CAPTION_RE.match(t) or re.match(r"^Table\b", t, re.IGNORECASE):
        return None
    m = _HYPHEN_CONTINUATION_PREFIX_RE.match(t)
    if not m:
        return None
    return m.group(1) + (m.group(2) or "")


def _extend_mention_text(
    para: str,
    page_idx: int,
    para_idx: int,
    paras: List[str],
    pages: List[str],
) -> str:
    """Rejoin hyphen breaks and incomplete sentences across paragraphs or pages."""
    out = para.strip()
    extensions = 0

    for nxt in _continuation_paragraphs(page_idx, para_idx, paras, pages):
        if extensions >= MAX_MENTION_EXTENSIONS:
            break

        nxt_text = nxt.strip()
        if not nxt_text:
            continue

        if out.rstrip().endswith("-"):
            merged_base = out.rstrip()[:-1]
            if _is_safe_continuation(nxt_text):
                out = _collapse_ws(merged_base + nxt_text)
                extensions += 1
                continue
            prefix = _extract_hyphen_continuation_prefix(nxt_text)
            if prefix:
                out = _collapse_ws(merged_base + prefix)
                break
            continue

        if _ends_complete_sentence(out) or out.rstrip().endswith(":"):
            break

        if not _is_safe_continuation(nxt_text):
            continue

        if extensions == 0 and not nxt_text[0].islower():
            break

        out = _collapse_ws(out + " " + nxt_text)
        extensions += 1

    return out


def _is_usable_mention(text: str) -> bool:
    t = text.strip()
    if len(t) < MIN_MENTION_CHARS:
        return False
    if re.fullmatch(r"-+", t):
        return False
    if _HEADING_ONLY_RE.match(t) and len(t) < 80:
        return False
    if t.startswith("$$") or t.startswith("\\["):
        return False
    url_hits = len(_FOOTNOTE_URL_RE.findall(t))
    if url_hits >= 2:
        return False
    if url_hits >= 1 and len(t) < 200:
        return False
    return True


def find_mentions(pages: List[str]) -> Dict[str, List[dict]]:
    """Map table number -> narrative paragraphs that cite it (+ nearby paragraphs)."""
    mentions: Dict[str, List[dict]] = {}
    seen: Dict[str, set] = {}

    for page_idx, page_text in enumerate(pages, start=1):
        paras = page_paragraphs_without_tables(page_text)
        for i, para in enumerate(paras):
            if CAPTION_RE.match(para):
                continue
            nums = _table_nums_in_para(para)
            if not nums:
                continue

            lo = max(0, i - ADJACENT_PARA_WINDOW)
            hi = min(len(paras), i + ADJACENT_PARA_WINDOW + 1)
            for j in range(lo, hi):
                candidate = paras[j]
                if CAPTION_RE.match(candidate):
                    continue
                if j != i:
                    adj_nums = _table_nums_in_para(candidate)
                    if adj_nums and adj_nums.isdisjoint(nums):
                        continue
                text = _extend_mention_text(candidate, page_idx, j, paras, pages)
                if not _is_usable_mention(text):
                    continue
                for n in nums:
                    key = _mention_key(page_idx, text)
                    seen.setdefault(n, set())
                    if key in seen[n]:
                        continue
                    seen[n].add(key)
                    mentions.setdefault(n, []).append({"page": page_idx, "text": text})

    for n in mentions:
        mentions[n].sort(key=lambda m: (m["page"], m["text"]))
    return mentions


# ---- Table HTML helpers ---------------------------------------------------
def header_and_rows_from_html(html: str) -> Tuple[List[str], List[str]]:
    """Split HTML table into (header_lines, body_lines)."""
    soup = BeautifulSoup(html, "html.parser")
    thead = soup.find("thead")
    tbody = soup.find("tbody")

    def _row_text(tr):
        cells = [c.get_text(separator=" ", strip=True) for c in tr.find_all(["th", "td"])]
        cells = [c for c in cells if c]
        return " | ".join(cells) if cells else None

    header_lines: List[str] = []
    body_lines: List[str] = []

    if thead is not None:
        for tr in thead.find_all("tr"):
            txt = _row_text(tr)
            if txt:
                header_lines.append(txt)

    trs = tbody.find_all("tr") if tbody is not None else soup.find_all("tr")
    for tr in trs:
        if thead is not None and tr in thead.find_all("tr"):
            continue
        cells = tr.find_all(["th", "td"])
        if not cells:
            continue
        txt = _row_text(tr)
        if not txt:
            continue
        if all(c.name == "th" for c in cells) and not body_lines:
            header_lines.append(txt)
        else:
            body_lines.append(txt)

    if not header_lines and body_lines:
        header_lines = [body_lines[0]]
        body_lines = body_lines[1:]
    return header_lines, body_lines


def table_header_text(html: str) -> str:
    """Header/column lines only — used for entity extraction (avoids model rows)."""
    return "\n".join(header_and_rows_from_html(html)[0])


def table_rows_text(html: str) -> str:
    """Flatten an HTML table into 'cell | cell' lines (header + body)."""
    header_lines, body_lines = header_and_rows_from_html(html)
    return "\n".join(header_lines + body_lines)


print("OCR + context helpers loaded")


In [ ]:
# ---- Normalization + GLiNER2 (benchmark string rules only) -------------------
import unicodedata


def _strip_accents(text: str) -> str:
    return "".join(ch for ch in unicodedata.normalize("NFKD", text) if not unicodedata.combining(ch))


def normalize_text(text: object) -> str:
    if text is None:
        return ""
    s = str(text).strip().lower()
    s = _strip_accents(s)
    return re.sub(r"\s+", " ", s)


def normalize_dataset(text: object) -> str:
    s = normalize_text(text)
    return s.replace(" ", "").replace("_", "").replace("-", "")


def _strip_trailing_plural(s: str) -> str:
    if not s or "@" in s or not s.isalpha():
        return s
    if len(s) > 3 and s.endswith("s") and not s.endswith("ss"):
        return s[:-1]
    return s


def normalize_metric(text: object) -> str:
    s = normalize_text(text)
    if not s:
        return ""
    s = re.sub(r"[^a-z0-9@]+", "", s)
    return _strip_trailing_plural(s)


def metric_dedup_key(raw: str) -> str:
    """Dedup key for metrics; alternate @-suffix spellings share the same key."""
    nm = normalize_metric(raw)
    if not nm:
        return ""
    m = re.match(r"^h@(\d+)$", nm)
    if m:
        return f"hits@{m.group(1)}"
    return nm


def _is_incomplete_metric_key(key: str) -> bool:
    """Normalized metric keys ending in @ without a numeric suffix (table header fragments)."""
    if not key or "@" not in key:
        return False
    suffix = key.split("@", 1)[1]
    return not suffix or not any(ch.isdigit() for ch in suffix)


def _pick_display(candidates: List[str]) -> str:
    """When several raw strings share a dedup key, keep the most informative one."""
    return max(candidates, key=lambda s: (len(s), any(c.isupper() for c in s)))


def _dedup_by_key(items: List[str], key_fn) -> List[str]:
    buckets: Dict[str, List[str]] = {}
    for value in items:
        key = key_fn(value)
        if not key:
            continue
        buckets.setdefault(key, []).append(value)
    return sorted(_pick_display(vals) for vals in buckets.values())


def _clean_entity(value: str) -> str:
    s = str(value).strip()
    s = re.sub(r"\[[^\]]{1,50}\]", "", s)
    s = re.sub(r"\((?:[^\)]*\d{4}[^\)]*|[^\)]*et\s*al\.?[^\)]*)\)", "", s, flags=re.IGNORECASE)
    s = re.sub(r"\$([^$]+?)\$", r"\1", s)
    s = re.sub(r"\\(?:textbf|textit|text|mathbf|mathrm|mathit)\{([^{}]*)\}", r"\1", s)
    for _ in range(3):
        s = re.sub(r"[_^]\{([^{}]*)\}", r"\1", s)
    s = s.replace("{", "").replace("}", "")
    s = re.sub(r"\s+", " ", s).strip(" .,:;-")
    return s


def _extract_entities_raw(text: str) -> Dict[str, Tuple[str, float]]:
    """GLiNER2 over one text chunk; {entity_text: (label, confidence)}."""
    best: Dict[str, Tuple[str, float]] = {}
    if not text or not text.strip():
        return best
    try:
        result = gliner2_model.extract_entities(
            text[:GLINER2_MAX_CHARS],
            GLINER2_LABEL_DESCRIPTIONS,
            include_confidence=True,
        )
    except Exception as e:
        print(f"  [gliner2 error] {type(e).__name__}: {e}")
        return best

    for label, items in (result or {}).get("entities", {}).items():
        if label not in GLINER2_LABELS:
            continue
        for item in items or []:
            if isinstance(item, dict):
                raw, score = str(item.get("text", "")), float(item.get("confidence", 1.0) or 1.0)
            else:
                raw, score = str(item), 1.0
            value = _clean_entity(raw)
            if score < GLINER2_MIN_SCORE or len(value) < 2:
                continue
            if re.fullmatch(r"[+-]?\d+(?:\.\d+)?", value):
                continue
            prev = best.get(value)
            if prev is None or score > prev[1]:
                best[value] = (label, score)
    return best


def _merge_best(target: Dict[str, Tuple[str, float]], other: Dict[str, Tuple[str, float]]) -> None:
    for value, (label, score) in other.items():
        prev = target.get(value)
        if prev is None or score > prev[1]:
            target[value] = (label, score)


def _keys_from_best(best: Dict[str, Tuple[str, float]], label: str, key_fn) -> set:
    keys = set()
    for value, (lbl, _) in best.items():
        if lbl != label:
            continue
        k = key_fn(value)
        if k:
            keys.add(k)
    return keys


def extract_table_entities(caption: str, html: str) -> Dict[str, List[str]]:
    """GLiNER on table HTML only (header + body rows). Caption is not sent to GLiNER."""
    header_lines, body_lines = header_and_rows_from_html(html)
    header_context = "\n".join(header_lines)

    table_best: Dict[str, Tuple[str, float]] = {}

    if header_context:
        _merge_best(table_best, _extract_entities_raw(header_context))

    for row_text in body_lines:
        prompt = (
            f"Table column headers: {header_context}\nRow: {row_text}"
            if header_context else row_text
        )
        _merge_best(table_best, _extract_entities_raw(prompt))

    if not table_best:
        full = "\n".join(header_lines + body_lines)
        _merge_best(table_best, _extract_entities_raw(full))

    # Caption-only keys (e.g. MRL loss name) — excluded from output.
    caption_only_ds: set = set()
    caption_only_mt: set = set()
    if caption:
        cap_best = _extract_entities_raw(caption)
        cap_ds = _keys_from_best(cap_best, "dataset", normalize_dataset)
        cap_mt = _keys_from_best(cap_best, "metric", metric_dedup_key)
        table_ds = _keys_from_best(table_best, "dataset", normalize_dataset)
        table_mt = _keys_from_best(table_best, "metric", metric_dedup_key)
        caption_only_ds = cap_ds - table_ds
        caption_only_mt = cap_mt - table_mt

    filtered_datasets: List[str] = []
    filtered_metrics: List[str] = []
    for value, (label, _) in table_best.items():
        if label == "model":
            continue
        if label == "dataset":
            k = normalize_dataset(value)
            if k and k not in caption_only_ds:
                filtered_datasets.append(value)
        elif label == "metric":
            k = metric_dedup_key(value)
            if k and not _is_incomplete_metric_key(k) and k not in caption_only_mt:
                filtered_metrics.append(value)

    return {
        "dataset": _dedup_by_key(filtered_datasets, normalize_dataset),
        "metric": _dedup_by_key(filtered_metrics, metric_dedup_key),
    }


print("GLiNER2 entity helpers loaded")

def _extract_entities_raw_gliner(text: str) -> Dict[str, Tuple[str, float]]:
    return _extract_entities_raw(text)


def extract_table_entities_gliner(caption: str, html: str) -> Dict[str, List[str]]:
    """Same as extract_table_entities (ollama notebook name)."""
    return extract_table_entities(caption, html)


In [ ]:
def _extract_json_object(text: str) -> Dict:
    text = text.strip()
    m = re.search(r"\{[\s\S]*\}", text)
    if not m:
        return {}
    blob = m.group(0)
    try:
        return json.loads(blob)
    except json.JSONDecodeError:
        return {}


_ORPHAN_PREFIX_RE = re.compile(
    r"^(?:reported|shown|presented|compared|listed|denoted|obtained|computed|evaluated)\b",
    re.IGNORECASE,
)


def _looks_orphaned_fragment(text: str) -> bool:
    t = text.strip()
    if not t:
        return True
    first = t[0]
    if first.islower() and _ORPHAN_PREFIX_RE.match(t):
        return True
    if first.islower() and re.search(r"\bin\s+the\s+table\s+\d+\b", t, re.IGNORECASE):
        return True
    return False


def _normalize_mention_text(text: str) -> str:
    return re.sub(r"\s+", " ", str(text).strip()).lower()


def _dedup_mentions(mentions: List[dict]) -> List[dict]:
    seen = set()
    out = []
    for m in mentions:
        page = m.get("page")
        text = str(m.get("text", "")).strip()
        if not isinstance(page, int) or not text:
            continue
        key = (page, _normalize_mention_text(text))
        if key in seen:
            continue
        seen.add(key)
        out.append({"page": page, "text": text})
    return out


def _mentions_explicitly_for_table(mentions: List[dict], table_num: Optional[str]) -> List[dict]:
    if not table_num:
        return _dedup_mentions(mentions)
    pat = re.compile(rf"\b(?:table|tab\.?)\s*{re.escape(table_num)}\b", re.IGNORECASE)
    out = []
    for m in mentions:
        text = str(m.get("text", "")).strip()
        page = m.get("page")
        if isinstance(page, int) and text and pat.search(text):
            out.append({"page": page, "text": text})
    return _dedup_mentions(out)


def _ollama_refine_mentions(
    table_label: str,
    caption: str,
    mentions: List[dict],
    *,
    ollama_model: str,
) -> List[dict]:
    if not mentions:
        return mentions

    table_num_match = re.search(r"\b(\d+(?:\.\d+)?)\b", table_label or "")
    table_num = table_num_match.group(1) if table_num_match else None

    instruction = {
        "task": "filter_mentions_for_one_table_keep_full_text",
        "strict_rules": [
            "Keep only mentions clearly about this specific table.",
            "Prefer mentions that explicitly cite the target table number.",
            "Drop mentions that are mainly about other tables.",
            "Do not shorten: keep full original mention text whenever possible.",
            "Do not return sentence fragments that start mid-thought (e.g., 'reported in the Table ...').",
            "Do not invent content.",
            "Preserve page numbers from input mentions.",
            "Output JSON only with key 'mentions'.",
        ],
        "target_table_label": table_label,
        "target_table_number": table_num,
        "target_caption": caption,
        "input_mentions": mentions,
        "output_schema": {"mentions": [{"page": "int", "text": "str"}]},
    }

    payload = {
        "model": ollama_model,
        "prompt": json.dumps(instruction, ensure_ascii=False),
        "stream": False,
        "options": {"temperature": 0},
    }

    try:
        r = requests.post(OLLAMA_URL, json=payload, timeout=OLLAMA_TIMEOUT)
        r.raise_for_status()
        response_text = r.json().get("response", "")
        data = _extract_json_object(response_text)
        if not isinstance(data, dict) or "mentions" not in data:
            return mentions  # parse failure fallback

        raw_mentions = data.get("mentions", [])
        clean = []
        for m in raw_mentions:
            if not isinstance(m, dict):
                continue
            page = m.get("page")
            text = str(m.get("text", "")).strip()
            if isinstance(page, int) and text and not _looks_orphaned_fragment(text):
                clean.append({"page": page, "text": text})

        clean = _dedup_mentions(clean)
        if clean:
            return clean

        # If LLM over-filters to empty, fallback to explicit Table N mentions.
        explicit = _mentions_explicitly_for_table(mentions, table_num)
        if explicit:
            return explicit
        return _dedup_mentions(mentions)
    except Exception:
        return _dedup_mentions(mentions)


In [ ]:
from pylatexenc.latex2text import LatexNodes2Text

_latex2text = LatexNodes2Text()
_LATEX_MATH_RE = re.compile(
    r"\$\$([^$]+)\$\$|\$([^$]+)\$|\\\(([^)]+)\\\)|\\\[([^\]]+)\\\]"
)


def _convert_latex_fragment(fragment: str) -> str:
    try:
        return _latex2text.latex_to_text(fragment)
    except Exception:
        return fragment


def latex_to_plain(text: str) -> str:
    """Convert inline/display LaTeX segments to readable Unicode; leave plain text intact."""
    if not text or not str(text).strip():
        return text

    def _repl(match: re.Match) -> str:
        fragment = next(g for g in match.groups() if g is not None)
        return _convert_latex_fragment(fragment)

    return _LATEX_MATH_RE.sub(_repl, str(text))


def _format_mentions_for_export(mentions: List[dict]) -> List[dict]:
    return [{"page": m["page"], "text": latex_to_plain(m["text"])} for m in mentions]


def extract_context_from_pages(
    pdf_path: Path,
    pages: List[str],
    *,
    use_ollama: bool = False,
    ollama_model: Optional[str] = None,
) -> dict:
    mentions_by_num = find_mentions(pages)

    tables = []
    for page_idx, page_text in enumerate(pages, start=1):
        paired = pair_captions_to_tables(page_text)
        for t_i, item in enumerate(paired, start=1):
            html = item["match"].group(0)
            caption_raw = item["caption"] or ""
            table_num = item["table_num"]
            table_label = f"Table {table_num}" if table_num else None
            mentions_raw = mentions_by_num.get(table_num, []) if table_num else []
            if use_ollama:
                if not ollama_model:
                    raise ValueError("ollama_model is required when use_ollama=True")
                mentions_raw = _ollama_refine_mentions(
                    table_label or "Table",
                    caption_raw,
                    mentions_raw,
                    ollama_model=ollama_model,
                )

            ents = extract_table_entities_gliner(caption_raw, html) if use_ollama else extract_table_entities(caption_raw, html)

            tables.append({
                "table_id": f"{pdf_path.stem}_p{page_idx}_t{t_i}",
                "table_label": table_label,
                "page": page_idx,
                "caption": latex_to_plain(caption_raw),
                "mentions": _format_mentions_for_export(mentions_raw),
                "datasets": ents["dataset"],
                "metrics": ents["metric"],
            })

    return {
        "paper": pdf_path.stem,
        "source_pdf": str(pdf_path.resolve()),
        "num_tables": len(tables),
        "tables": tables,
    }


def extract_context_for_pdf(
    pdf_path: Path,
    *,
    use_ollama: bool = False,
    ollama_model: Optional[str] = None,
    force_ocr: bool = False,
) -> dict:
    tag = f"heuristic_ollama/{ollama_model}" if use_ollama else "heuristic"
    print("\n" + "=" * 70)
    print(f"Processing ({tag}): {pdf_path.name}")
    print("=" * 70)
    pages = ocr_pdf_pages(pdf_path, force_ocr=force_ocr)
    doc = extract_context_from_pages(
        pdf_path,
        pages,
        use_ollama=use_ollama,
        ollama_model=ollama_model,
    )
    t = doc["tables"]
    print(
        f"  tables={len(t)} | captions={sum(1 for x in t if x['caption'])} | "
        f"datasets={sum(len(x['datasets']) for x in t)} | metrics={sum(len(x['metrics']) for x in t)}"
    )
    return doc

def collect_pdf_paths(input_path: Path) -> List[Path]:
    input_path = input_path.resolve()
    if input_path.is_file():
        if input_path.suffix.lower() != ".pdf":
            raise ValueError(f"Not a PDF file: {input_path}")
        return [input_path]
    if input_path.is_dir():
        pdfs = sorted(input_path.glob("*.pdf"))
        if not pdfs:
            raise FileNotFoundError(f"No PDF files in {input_path}")
        return pdfs
    raise FileNotFoundError(f"Input path does not exist: {input_path}")


def extract_documents(
    input_path: Path,
    *,
    force_ocr: bool = False,
    use_ollama: bool = False,
    ollama_model: Optional[str] = None,
) -> List[dict]:
    return [
        extract_context_for_pdf(
            pdf_path,
            use_ollama=use_ollama,
            ollama_model=ollama_model,
            force_ocr=force_ocr,
        )
        for pdf_path in collect_pdf_paths(input_path)
    ]


def write_result_json(
    documents: List[dict],
    output_path: Path,
    *,
    pdf_dir: Path,
    description: str,
) -> dict:
    result_json = {
        "description": description,
        "pdf_dir": str(pdf_dir.resolve()),
        "num_documents": len(documents),
        "total_tables": sum(d["num_tables"] for d in documents),
        "documents": documents,
    }
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with output_path.open("w", encoding="utf-8") as f:
        json.dump(result_json, f, indent=2, ensure_ascii=False)
    return result_json


In [ ]:
# --- Step 1: extraction (heuristic + heuristic_ollama per model) ---
for model in OLLAMA_MODELS:
    health_payload = {
        "model": model,
        "prompt": '{"ok": true}',
        "stream": False,
        "options": {"temperature": 0},
    }
    resp = requests.post(OLLAMA_URL, json=health_payload, timeout=OLLAMA_TIMEOUT)
    resp.raise_for_status()
    print(f"Ollama OK [{model}]:", resp.json().get("response", "")[:80])

gliner_documents = []
ollama_documents_by_model = {model: [] for model in OLLAMA_MODELS}
skipped_pdfs = []

for pdf_path in PDF_FILES:
    print("\n" + "=" * 70)
    print(f"Processing: {pdf_path.name}")
    print("=" * 70)

    try:
        pages = ocr_pdf_pages(pdf_path, force_ocr=FORCE_OCR)
    except Exception as e:
        print(f"  SKIP (OCR failed): {type(e).__name__}: {e}")
        skipped_pdfs.append({"pdf": pdf_path.name, "error": str(e)})
        continue

    doc_gliner = extract_context_from_pages(pdf_path, pages, use_ollama=False)
    gliner_documents.append(doc_gliner)
    t = doc_gliner["tables"]
    print(
        f"  [{EVAL_MODE_HEURISTIC}] tables={len(t)} | captions={sum(1 for x in t if x['caption'])} | "
        f"datasets={sum(len(x['datasets']) for x in t)} | metrics={sum(len(x['metrics']) for x in t)}"
    )

    for model in OLLAMA_MODELS:
        doc_ollama = extract_context_from_pages(
            pdf_path,
            pages,
            use_ollama=True,
            ollama_model=model,
        )
        ollama_documents_by_model[model].append(doc_ollama)
        t = doc_ollama["tables"]
        print(
            f"  [{EVAL_MODE_HEURISTIC_OLLAMA}/{model}] tables={len(t)} | captions={sum(1 for x in t if x['caption'])} | "
            f"datasets={sum(len(x['datasets']) for x in t)} | metrics={sum(len(x['metrics']) for x in t)}"
        )

gliner_result = write_result_json(
    gliner_documents,
    GLINER_OUTPUT_PATH,
    pdf_dir=PDF_DIR,
    description=GLINER_RESULT_JSON_DESCRIPTION,
)

ollama_results = {}
for model, documents in ollama_documents_by_model.items():
    out_path = ollama_output_path(model)
    ollama_results[model] = write_result_json(
        documents,
        out_path,
        pdf_dir=PDF_DIR,
        description=ollama_result_description(model),
    )

raw_docs = {d["paper"]: d for d in gliner_documents}
ollama_docs_by_model = {
    model: {d["paper"]: d for d in documents}
    for model, documents in ollama_documents_by_model.items()
}

print("\n" + "=" * 70)
print("DONE extraction")
print("=" * 70)
print(f"  JSON ({EVAL_MODE_HEURISTIC}): {GLINER_OUTPUT_PATH}")
for model in OLLAMA_MODELS:
    print(f"  JSON ({EVAL_MODE_HEURISTIC_OLLAMA}, {model}): {ollama_output_path(model)}")
print(
    f"  documents: {gliner_result['num_documents']} | "
    f"tables: {EVAL_MODE_HEURISTIC}={gliner_result['total_tables']}"
)
for model in OLLAMA_MODELS:
    print(f"    {EVAL_MODE_HEURISTIC_OLLAMA}/{model}: {ollama_results[model]['total_tables']} tables")
if skipped_pdfs:
    print(f"\nSkipped {len(skipped_pdfs)} unreadable PDF(s):")
    for row in skipped_pdfs:
        print(f"  - {row['pdf']}: {row['error']}")

In [ ]:
# --- Evaluation helpers ---

def _table_match_key(paper: str, table: dict):
    table_id = table.get("table_id")
    if table_id:
        return (paper, table_id)
    label = table.get("table_label")
    if label:
        return (paper, label)
    return None


def _docs_to_index(docs):
    idx = {}
    for paper, doc in docs.items():
        for t in doc.get("tables", []):
            key = _table_match_key(paper, t)
            if key:
                idx[key] = t
    return idx


def load_ground_truth(path: Path):
    payload = json.loads(path.read_text(encoding="utf-8"))
    return _docs_to_index({d["paper"]: d for d in payload["documents"]})


_bert_scorer = None

def get_bert_scorer():
    global _bert_scorer
    if _bert_scorer is None:
        from bert_score import BERTScorer
        _bert_scorer = BERTScorer(lang=BERTSCORE_LANG, device=BERTSCORE_DEVICE)
    return _bert_scorer


def bertscore_concat(pred_items, ref_items):
    preds = [x.strip() for x in pred_items if str(x).strip()]
    refs = [x.strip() for x in ref_items if str(x).strip()]
    if not preds and not refs:
        return 1.0, 1.0, 1.0
    if not preds or not refs:
        return 0.0, 0.0, 0.0
    P, R, F1 = get_bert_scorer().score([MENTION_SEP.join(preds)], [MENTION_SEP.join(refs)])
    return float(P[0]), float(R[0]), float(F1[0])


def bertscore_best_match(pred_items, ref_items):
    preds = [x.strip() for x in pred_items if str(x).strip()]
    refs = [x.strip() for x in ref_items if str(x).strip()]
    if not preds and not refs:
        return 1.0, 1.0, 1.0
    if not preds or not refs:
        return 0.0, 0.0, 0.0
    p_scores = []
    for p in preds:
        _, _, F1 = get_bert_scorer().score([p] * len(refs), refs)
        p_scores.append(float(F1.max()))
    precision = sum(p_scores) / len(p_scores)
    r_scores = []
    for r in refs:
        _, _, F1 = get_bert_scorer().score(preds, [r] * len(preds))
        r_scores.append(float(F1.max()))
    recall = sum(r_scores) / len(r_scores)
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0
    return precision, recall, f1


def set_prf1(pred_values, ref_values, key_fn):
    pred_set = {key_fn(v) for v in pred_values if key_fn(v)}
    ref_set = {key_fn(v) for v in ref_values if key_fn(v)}
    if not pred_set and not ref_set:
        return 1.0, 1.0, 1.0
    if not pred_set or not ref_set:
        return 0.0, 0.0, 0.0
    tp = len(pred_set & ref_set)
    p = tp / len(pred_set)
    r = tp / len(ref_set)
    f1 = (2 * p * r / (p + r)) if (p + r) else 0.0
    return p, r, f1


def mention_texts(table):
    return [str(m.get("text", "")).strip() for m in table.get("mentions", []) if str(m.get("text", "")).strip()]


CONTEXT_EVAL_METRIC_COLS = ["mentions_p", "mentions_r", "mentions_f1"]


def evaluate_mode(name, pred_index, gt_index, *, ollama_model: str = ""):
    rows = []
    for paper, match_id in sorted(set(gt_index) | set(pred_index)):
        gt_t = gt_index.get((paper, match_id))
        pred_t = pred_index.get((paper, match_id))
        gt_mentions = mention_texts(gt_t or {})
        pred_mentions = mention_texts(pred_t or {})
        men_p, men_r, men_f1 = bertscore_best_match(pred_mentions, gt_mentions)
        rows.append({
            "mode": name,
            "ollama_model": ollama_model,
            "paper": paper,
            "table_id": (gt_t or pred_t or {}).get("table_id") or match_id,
            "table_label": (gt_t or pred_t or {}).get("table_label") or "",
            "in_gt": gt_t is not None,
            "in_pred": pred_t is not None,
            "mentions_p": men_p,
            "mentions_r": men_r,
            "mentions_f1": men_f1,
            "n_gt_mentions": len(gt_mentions),
            "n_pred_mentions": len(pred_mentions),
        })
    return pd.DataFrame(rows)


def macro_summary(df, group_cols=("mode", "ollama_model")):
    cols = [c for c in CONTEXT_EVAL_METRIC_COLS if c in df.columns]
    return df.groupby(list(group_cols))[cols].mean(numeric_only=True).reset_index()


In [ ]:
# --- Step 2: mentions evaluation (BERTScore best-match; caption omitted — same in all modes) ---
gt_index = load_ground_truth(GROUND_TRUTH_PATH)
raw_index = _docs_to_index(raw_docs)

print("Eval metrics: mentions_p/r/f1 only (caption skipped — identical for heuristic vs heuristic_ollama).")
print(f"Ground truth tables: {len(gt_index)}")
print(f"{EVAL_MODE_HEURISTIC}: {len(raw_index)} tables")
for model in OLLAMA_MODELS:
    idx = _docs_to_index(ollama_docs_by_model[model])
    print(f"{EVAL_MODE_HEURISTIC_OLLAMA} [{model}]: {len(idx)} tables")

frames = [evaluate_mode(EVAL_MODE_HEURISTIC, raw_index, gt_index)]
for model in OLLAMA_MODELS:
    ollama_index = _docs_to_index(ollama_docs_by_model[model])
    frames.append(
        evaluate_mode(
            EVAL_MODE_HEURISTIC_OLLAMA,
            ollama_index,
            gt_index,
            ollama_model=model,
        )
    )

df_all = pd.concat(frames, ignore_index=True)

summary = macro_summary(df_all)
display(summary)

out_summary = EVAL_OUTPUT_DIR / "context_eval.csv"
summary.to_csv(out_summary, index=False)
print(f"Saved {out_summary}")